In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
train = pd.read_csv("../data/raw/train.csv")

test = pd.read_csv("../data/raw/test.csv")

In [3]:
train['NetworkScore'] = train['NetworkScore'].fillna(
    train['NetworkScore'].median()
)

train['Age'] = train['Age'].fillna(
    train['Age'].median()
)

train['IsActiveMember'] = train[
    'IsActiveMember'
].fillna(
    train['IsActiveMember'].mode()[0]
)

train['EstimatedMonthlyUsage'] = train[
    'EstimatedMonthlyUsage'
].fillna(
    train['EstimatedMonthlyUsage'].median()
)

In [4]:
test['NetworkScore'] = test['NetworkScore'].fillna(
    test['NetworkScore'].median()
)

test['Age'] = test['Age'].fillna(
    test['Age'].median()
)

test['IsActiveMember'] = test[
    'IsActiveMember'
].fillna(
    test['IsActiveMember'].mode()[0]
)

test['EstimatedMonthlyUsage'] = test[
    'EstimatedMonthlyUsage'
].fillna(
    test['EstimatedMonthlyUsage'].median()
)

In [5]:
train = train.drop(
    ['CustomerID','Surname'],
    axis=1
)

test = test.drop(
    ['CustomerID','Surname'],
    axis=1
)

In [6]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

for col in ['Region','Gender']:

    train[col] = encoder.fit_transform(
        train[col]
    )

    test[col] = encoder.transform(
        test[col]
    )

In [7]:
train['CustomerValue'] = (
    train['MonthlyCharge']
    *
    train['NumOfProducts']
)

test['CustomerValue'] = (
    test['MonthlyCharge']
    *
    test['NumOfProducts']
)

In [8]:
train['UsagePerProduct'] = (
    train['EstimatedMonthlyUsage']
    /
    (train['NumOfProducts']+1)
)

test['UsagePerProduct'] = (
    test['EstimatedMonthlyUsage']
    /
    (test['NumOfProducts']+1)
)

In [9]:
train['TenureGroup'] = pd.cut(
    train['Tenure'],
    bins=[-1,2,5,10,100],
    labels=[0,1,2,3]
)

test['TenureGroup'] = pd.cut(
    test['Tenure'],
    bins=[-1,2,5,10,100],
    labels=[0,1,2,3]
)

train['TenureGroup'] = \
train['TenureGroup'].astype(int)

test['TenureGroup'] = \
test['TenureGroup'].astype(int)

In [10]:
train.select_dtypes(
include='object'
).columns

Index([], dtype='str')

In [11]:
X = train.drop(
    'Exited',
    axis=1
)

y = train['Exited']

In [12]:
X_train,\
X_valid,\
y_train,\
y_valid = train_test_split(

X,
y,

test_size=0.20,

random_state=42,

stratify=y
)

In [13]:
lr = LogisticRegression(
    max_iter=1000
)

lr.fit(
    X_train,
    y_train
)

pred_lr = lr.predict(
    X_valid
)

c:\Users\shlok\OneDrive\Desktop\python project\telecom-customer-churn\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_valid_scaled = scaler.transform(
    X_valid
)

In [15]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(

max_iter=3000,

random_state=42

)

lr.fit(

X_train_scaled,

y_train

)

pred_lr = lr.predict(

X_valid_scaled

)

In [16]:
from sklearn.metrics import \
accuracy_score,\
precision_score,\
recall_score,\
f1_score

print(
"Accuracy:",
accuracy_score(
y_valid,
pred_lr
)
)

print(
"Precision:",
precision_score(
y_valid,
pred_lr
)
)

print(
"Recall:",
recall_score(
y_valid,
pred_lr
)
)

print(
"F1:",
f1_score(
y_valid,
pred_lr
)
)

Accuracy: 0.7189054726368159
Precision: 0.7
Recall: 0.735897435897436
F1: 0.7175


In [17]:
rf = RandomForestClassifier(

n_estimators=300,

max_depth=8,

random_state=42

)

rf.fit(
X_train,
y_train
)

pred_rf = rf.predict(
X_valid
)

In [18]:
print(
accuracy_score(
y_valid,
pred_rf
)
)

0.7512437810945274


In [19]:
xgb = XGBClassifier(

n_estimators=300,

learning_rate=0.05,

max_depth=5,

random_state=42

)

xgb.fit(
X_train,
y_train
)

pred_xgb = xgb.predict(
X_valid
)

In [20]:
xgb = XGBClassifier(

n_estimators=300,

learning_rate=0.05,

max_depth=5,

random_state=42

)

xgb.fit(
X_train,
y_train
)

pred_xgb = xgb.predict(
X_valid
)

In [21]:
print(
accuracy_score(
y_valid,
pred_xgb
)
)

0.7524875621890548


In [22]:
train['ChargePerTenure'] = (
    train['MonthlyCharge']
    /
    (train['Tenure']+1)
)

test['ChargePerTenure'] = (
    test['MonthlyCharge']
    /
    (test['Tenure']+1)
)

In [23]:
train['UsageIntensity'] = (
    train['EstimatedMonthlyUsage']
    /
    train['NetworkScore']
)

test['UsageIntensity'] = (
    test['EstimatedMonthlyUsage']
    /
    test['NetworkScore']
)

In [24]:
train['ProductPerTenure'] = (
    train['NumOfProducts']
    /
    (train['Tenure']+1)
)

test['ProductPerTenure'] = (
    test['NumOfProducts']
    /
    (test['Tenure']+1)
)

In [25]:
X = train.drop(
'Exited',
axis=1
)

y = train['Exited']

X_train,\
X_valid,\
y_train,\
y_valid = train_test_split(

X,
y,

test_size=0.20,

random_state=42,

stratify=y
)

In [26]:
from xgboost import XGBClassifier

xgb = XGBClassifier(

n_estimators=800,

learning_rate=0.03,

max_depth=7,

subsample=0.8,

colsample_bytree=0.8,

min_child_weight=3,

random_state=42

)

xgb.fit(
X_train,
y_train
)

pred = xgb.predict(
X_valid
)

print(
accuracy_score(
y_valid,
pred
)
)

0.7549751243781094


In [27]:
train['AgeTenureRatio'] = (
train['Age']
/
(train['Tenure']+1)
)

test['AgeTenureRatio'] = (
test['Age']
/
(test['Tenure']+1)
)

In [28]:
train['ChargeUsageRatio'] = (
train['MonthlyCharge']
/
(train['EstimatedMonthlyUsage']+1)
)

test['ChargeUsageRatio'] = (
test['MonthlyCharge']
/
(test['EstimatedMonthlyUsage']+1)
)

In [29]:
train['EngagementScore'] = (

train['IsActiveMember']

+

train['HasInternetService']

+

train['NumOfProducts']

)

test['EngagementScore'] = (

test['IsActiveMember']

+

test['HasInternetService']

+

test['NumOfProducts']

)

In [30]:
X = train.drop(
'Exited',
axis=1
)

y = train['Exited']

In [31]:
from xgboost import XGBClassifier

xgb = XGBClassifier(

n_estimators=1200,

learning_rate=0.02,

max_depth=8,

subsample=0.85,

colsample_bytree=0.85,

gamma=0.1,

min_child_weight=2,

random_state=42

)

xgb.fit(
X_train,
y_train
)

pred = xgb.predict(
X_valid
)

print(
accuracy_score(
y_valid,
pred
)
)

0.7512437810945274
